# 04. Train The Student With MLX-LM

This notebook trains the small MLX student on the teacher trajectories from notebook 03.

The training objective is offline token-level imitation: given the conversation history and tool schemas, train the student to produce the teacher's next assistant message. We mask the prompt, so loss is only computed on the final teacher response.


In [1]:
from __future__ import annotations

from statistics import mean
import json
import os
from pathlib import Path
import sys

cwd = Path.cwd()
ROOT = cwd if (cwd / "common" / "config.py").exists() else cwd.parent
if not (ROOT / "common" / "config.py").exists():
    raise RuntimeError("Run this notebook from the repo root or a blog folder under the repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common import config as cfg
from common import mlx_resumable_lora
from common import sft_rows

paths = cfg.setup_notebook_paths(blog_dir_name="1-distilling-a-0-8b-tool-calling-agent")
ROOT = paths.root
BLOG_DIR = paths.blog_dir
DATA_DIR = paths.data_dir
OUTPUT_DIR = paths.output_dir
ENV_PATH = paths.env_path
DOTENV_LOADED = paths.dotenv_loaded

TAU_BENCH_USER_SIMULATOR_LLM = cfg.required_env("TAU_BENCH_USER_SIMULATOR_LLM")
USER_SIMULATOR_SLUG = cfg.filename_slug(TAU_BENCH_USER_SIMULATOR_LLM)
teacher_config = cfg.teacher_config_from_env(default_provider="vllm_raw", default_model=cfg.TEACHER_MODEL)
TEACHER_MODEL = teacher_config.model_name
TEACHER_SLUG = cfg.filename_slug(TEACHER_MODEL)
TEACHER_PROVIDER = teacher_config.provider
STUDENT_MODEL = cfg.STUDENT_MODEL
STUDENT_SLUG = cfg.filename_slug(STUDENT_MODEL)

SFT_WORK_DIR = OUTPUT_DIR / f"{STUDENT_SLUG}_tau3_retail_sft_mlx_lm"
MLX_DATA_DIR = SFT_WORK_DIR / "mlx_lm_data"
ADAPTER_OUTPUT_DIR = SFT_WORK_DIR / f"{STUDENT_SLUG}_tau3_retail_mlx_lora_adapter"
SFT_SOURCE_PATH = OUTPUT_DIR / f"{TEACHER_SLUG}_{TEACHER_PROVIDER}_tau3_bench_retail_train_successful_sft_chat_rows_{USER_SIMULATOR_SLUG}.jsonl"

print("Project root:", ROOT)
print("Loaded .env:", DOTENV_LOADED, "from", ENV_PATH)
print("SFT source path:", SFT_SOURCE_PATH)
print("Work dir:", SFT_WORK_DIR)


Project root: /Users/kargarisaac/codes/personal/distillation-blogs
Loaded .env: True from /Users/kargarisaac/codes/personal/distillation-blogs/.env
SFT source path: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/mlx_community_qwen3_5_35b_a3b_8bit_vllm_raw_tau3_bench_retail_train_successful_sft_chat_rows_openai_gpt_5_4_mini.jsonl
Work dir: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/mlx_community_qwen3_5_0_8b_mlx_bf16_tau3_retail_sft_mlx_lm


## Load Teacher Rows

Run notebook 03 first. It creates `messages + tools` JSONL rows from successful teacher trajectories.


In [2]:
if not SFT_SOURCE_PATH.exists():
    candidates = sorted(OUTPUT_DIR.glob("*_tau3_bench_retail_train_successful_sft_chat_rows_*.jsonl"))
    print("Could not find the expected SFT source path.")
    print("Expected:", SFT_SOURCE_PATH)
    print("Candidate SFT files:")
    for candidate in candidates:
        print(" -", candidate)
    raise FileNotFoundError(SFT_SOURCE_PATH)

rows = cfg.load_jsonl(SFT_SOURCE_PATH)
if not rows:
    raise RuntimeError("The SFT source file exists but contains no rows.")

print("Loaded SFT rows:", len(rows))
print("Unique task ids:", len({row['task_id'] for row in rows}))
print("Tool-call target rows:", sum(row.get("is_tool_call_target", False) for row in rows))
print("Natural-language target rows:", sum(not row.get("is_tool_call_target", False) for row in rows))
print("First row roles:", [message["role"] for message in rows[0]["messages"][-5:]])
print("\nFirst target preview:\n", rows[0]["target_text"][:800])


Loaded SFT rows: 363
Unique task ids: 31
Tool-call target rows: 193
Natural-language target rows: 170
First row roles: ['system', 'assistant', 'user', 'assistant']

First target preview:
 <tool_call>
<function=get_order_details>
<parameter=order_id>
#W2378156
</parameter>
</function>
</tool_call>


## Measure Sequence Lengths

MLX-LM will apply Qwen's chat template and include the retail tools in the rendered prompt. We measure that exact training sequence here.

The first verified Mac-native smoke config is conservative:

- student base: `mlx-community/Qwen3.5-0.8B-MLX-bf16` (bf16, non-quantized)
- LoRA layers: 1
- batch size: 1
- prompt masking: on
- tested length: about 4k tokens in the smoke run
- memory observed in the smoke run: about 8.6 GB peak

Longer rows are real data, but they need a stronger verified context/memory setting. We do not silently truncate them for the first run. If memory becomes the blocker, the 4-bit MLX checkpoint is a fallback, not the default path.


In [3]:
tokenizer = cfg.load_tokenizer()
length_rows = []
for index, row in enumerate(rows):
    token_count = sft_rows.mlx_chat_row_token_length(row, tokenizer)
    length_rows.append({
        "index": index,
        "task_id": row["task_id"],
        "source_message_index": row["source_message_index"],
        "tokens": token_count,
        "is_tool_call_target": row.get("is_tool_call_target", False),
    })

lengths = [item["tokens"] for item in length_rows]
length_stats = {
    "min": min(lengths),
    "p50": cfg.percentile_int(lengths, 0.50),
    "p90": cfg.percentile_int(lengths, 0.90),
    "p95": cfg.percentile_int(lengths, 0.95),
    "max": max(lengths),
    "mean": round(mean(lengths), 1),
}
print(json.dumps(length_stats, indent=2))
print()
print("Longest rows:")
for item in sorted(length_rows, key=lambda item: item["tokens"], reverse=True)[:10]:
    print(item)


{
  "min": 4938,
  "p50": 6184,
  "p90": 10094,
  "p95": 11081,
  "max": 15331,
  "mean": 6869.9
}

Longest rows:
{'index': 92, 'task_id': '20', 'source_message_index': 32, 'tokens': 15331, 'is_tool_call_target': False}
{'index': 91, 'task_id': '20', 'source_message_index': 30, 'tokens': 14509, 'is_tool_call_target': True}
{'index': 90, 'task_id': '20', 'source_message_index': 28, 'tokens': 14316, 'is_tool_call_target': False}
{'index': 89, 'task_id': '20', 'source_message_index': 26, 'tokens': 14004, 'is_tool_call_target': False}
{'index': 172, 'task_id': '54', 'source_message_index': 32, 'tokens': 13171, 'is_tool_call_target': False}
{'index': 339, 'task_id': '95', 'source_message_index': 36, 'tokens': 12409, 'is_tool_call_target': False}
{'index': 25, 'task_id': '7', 'source_message_index': 34, 'tokens': 12235, 'is_tool_call_target': False}
{'index': 171, 'task_id': '54', 'source_message_index': 30, 'tokens': 12214, 'is_tool_call_target': True}
{'index': 41, 'task_id': '8', 'source_

## Choose The First MLX Training Config

This is intentionally notebook-local configuration, not `.env` configuration. `.env` should hold external/runtime settings such as API keys and server URLs. The training knobs below are part of the experiment, so we keep them visible in the notebook.

This cell makes the memory tradeoff explicit. If you want to attempt longer-context training, change `MAX_SEQ_LENGTH` and rerun. Rows longer than the selected maximum are dropped rather than silently truncated.


In [4]:
MAX_SEQ_LENGTH = 16500
LORA_NUM_LAYERS = 1
LORA_RANK = 8
LORA_SCALE = 20.0
LEARNING_RATE = 1e-5
BATCH_SIZE = 4
ITERS = -1  # -1 means one pass over the kept rows, measured in batches.
GRAD_ACCUMULATION_STEPS = 1
VALIDATION_TASK_FRACTION = 0.10
SPLIT_SEED = 42
TRAINING_SEED = 0
RESUME_TRAINING = "none"  # Use "latest" only when resuming with the same batch/config.
CLEAR_CACHE_THRESHOLD = 0
GRAD_CHECKPOINT = True

rows_that_fit = [row for row, meta in zip(rows, length_rows) if meta["tokens"] <= MAX_SEQ_LENGTH]
rows_too_long = [meta for meta in length_rows if meta["tokens"] > MAX_SEQ_LENGTH]

print("Student MLX model:", STUDENT_MODEL)
print("Max sequence length:", MAX_SEQ_LENGTH)
print("LoRA layers:", LORA_NUM_LAYERS)
print("LoRA rank:", LORA_RANK)
print("LoRA scale:", LORA_SCALE)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation steps:", GRAD_ACCUMULATION_STEPS)
print("Gradient checkpointing:", GRAD_CHECKPOINT)
print("Training seed:", TRAINING_SEED)
print("Resume mode:", RESUME_TRAINING)
print("Rows kept:", len(rows_that_fit))
print("Rows longer than max sequence length:", len(rows_too_long))

if rows_too_long:
    print("\nLongest dropped rows:")
    for item in sorted(rows_too_long, key=lambda item: item["tokens"], reverse=True)[:10]:
        print(item)

if not rows_that_fit:
    raise RuntimeError("No rows fit the current MAX_SEQ_LENGTH. Increase MAX_SEQ_LENGTH or collect shorter rows.")

Student MLX model: mlx-community/Qwen3.5-0.8B-MLX-bf16
Max sequence length: 16500
LoRA layers: 1
LoRA rank: 8
LoRA scale: 20.0
Learning rate: 1e-05
Batch size: 4
Gradient accumulation steps: 1
Gradient checkpointing: True
Training seed: 0
Resume mode: none
Rows kept: 363
Rows longer than max sequence length: 0


## Split By Task And Write MLX-LM Data Files

We split by task id, not by row. That prevents later turns from the same solved trajectory leaking into validation.


In [ ]:
train_rows, validation_rows, validation_task_ids = sft_rows.split_sft_rows_by_task_id(
    rows_that_fit,
    validation_task_fraction=VALIDATION_TASK_FRACTION,
    seed=SPLIT_SEED,
)
if not validation_rows:
    validation_rows = train_rows[:1]

MLX_DATA_DIR.mkdir(parents=True, exist_ok=True)
cfg.write_jsonl(MLX_DATA_DIR / "train.jsonl", train_rows)
cfg.write_jsonl(MLX_DATA_DIR / "valid.jsonl", validation_rows)
cfg.write_jsonl(MLX_DATA_DIR / "test.jsonl", validation_rows)

print("Train rows:", len(train_rows))
print("Validation rows:", len(validation_rows))
print("Validation task ids:", sorted(validation_task_ids))
print("MLX data dir:", MLX_DATA_DIR)


Train rows: 338
Validation rows: 25
Validation task ids: ['46', '57', '73']
MLX data dir: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/mlx_community_qwen3_5_0_8b_mlx_bf16_tau3_retail_sft_mlx_lm/mlx_lm_data


: 

## Train With A Resumable MLX LoRA Runner

`mask_prompt=True` is the important training choice. It tells the trainer to compute loss only on the final assistant response in each row, while still letting the model read the full prompt.

Each checkpoint saves adapter weights, optimizer state, MLX random state, gradient accumulation state, global iteration, token counts, loss history, and a dataset fingerprint. To start from scratch, keep `RESUME_TRAINING = "none"`. To continue after an interruption, set `RESUME_TRAINING = "latest"` before rerunning this training cell. Keep batch size and related training config unchanged when resuming.

The memory log prints active, cache, and peak memory separately. Peak memory is a high-water mark, so it naturally tends to increase as longer batches appear; it is not the same as current live memory.


In [ ]:
if ITERS <= 0:
    ITERS = max(1, len(train_rows) // max(1, BATCH_SIZE))

CHECKPOINT_EVERY = max(1, min(25, ITERS))
STEPS_PER_EVAL = max(1, min(25, ITERS))
STEPS_PER_REPORT = 1
VAL_BATCHES = 1
RUN_TRAINING = True
ADAPTER_CONFIG_PATH = ADAPTER_OUTPUT_DIR / "adapter_config.json"
CHECKPOINT_ROOT = ADAPTER_OUTPUT_DIR / "training_checkpoints"
TRAINING_HISTORY_PATH = ADAPTER_OUTPUT_DIR / "training_history.jsonl"

training_config = mlx_resumable_lora.ResumableLoraConfig(
    model=STUDENT_MODEL,
    data_dir=MLX_DATA_DIR,
    adapter_path=ADAPTER_OUTPUT_DIR,
    total_iters=ITERS,
    max_seq_length=MAX_SEQ_LENGTH,
    batch_size=BATCH_SIZE,
    grad_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    val_batches=VAL_BATCHES,
    steps_per_report=STEPS_PER_REPORT,
    steps_per_eval=STEPS_PER_EVAL,
    save_every=CHECKPOINT_EVERY,
    num_layers=LORA_NUM_LAYERS,
    lora_rank=LORA_RANK,
    lora_scale=LORA_SCALE,
    mask_prompt=True,
    seed=TRAINING_SEED,
    grad_checkpoint=GRAD_CHECKPOINT,
    clear_cache_threshold=CLEAR_CACHE_THRESHOLD,
    resume=RESUME_TRAINING,
)

print("Training config:")
print(json.dumps(cfg.make_json_safe(training_config.__dict__), indent=2))
print("Adapter output dir:", ADAPTER_OUTPUT_DIR)
print("Checkpoint root:", CHECKPOINT_ROOT)
print("Training history path:", TRAINING_HISTORY_PATH)
print("Run training:", RUN_TRAINING)

if RUN_TRAINING:
    os.environ["HF_HUB_DISABLE_XET"] = "1"
    training_result = mlx_resumable_lora.run_resumable_lora_training(training_config)
else:
    print("Skipping training because RUN_TRAINING is False")
    training_result = {
        "adapter_path": str(ADAPTER_OUTPUT_DIR),
        "checkpoint_root": str(CHECKPOINT_ROOT),
        "history_path": str(TRAINING_HISTORY_PATH),
        "global_step": 0,
        "trained_tokens": 0,
        "train_history": [],
        "val_history": [],
    }

print("Training result:")
print(json.dumps(cfg.make_json_safe(training_result), indent=2)[:4000])


Training config:
{
  "model": "mlx-community/Qwen3.5-0.8B-MLX-bf16",
  "data_dir": "/Users/kargarisaac/codes/personal/distillation-blogs/outputs/mlx_community_qwen3_5_0_8b_mlx_bf16_tau3_retail_sft_mlx_lm/mlx_lm_data",
  "adapter_path": "/Users/kargarisaac/codes/personal/distillation-blogs/outputs/mlx_community_qwen3_5_0_8b_mlx_bf16_tau3_retail_sft_mlx_lm/mlx_community_qwen3_5_0_8b_mlx_bf16_tau3_retail_mlx_lora_adapter",
  "total_iters": 84,
  "max_seq_length": 16500,
  "batch_size": 4,
  "grad_accumulation_steps": 1,
  "learning_rate": 1e-05,
  "optimizer": "adam",
  "optimizer_config": {},
  "lr_schedule": null,
  "val_batches": 1,
  "steps_per_report": 1,
  "steps_per_eval": 25,
  "save_every": 25,
  "num_layers": 1,
  "lora_rank": 8,
  "lora_scale": 20.0,
  "lora_dropout": 0.0,
  "fine_tune_type": "lora",
  "mask_prompt": true,
  "seed": 0,
  "grad_checkpoint": true,
  "clear_cache_threshold": 0,
  "resume": "none"
}
Adapter output dir: /Users/kargarisaac/codes/personal/distillation

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Trainable parameters: 0.027% (0.201M/752.392M)
Loading datasets
Starting from scratch.
Starting training at iteration 1; target iteration 84.
Iter 1: Val loss 0.792, Val took 3.230s
Iter 1: Train loss 0.884, Learning Rate 1.000e-05, It/sec 0.008, Tokens/sec 1.856, Trained Tokens 233, Active mem 1.510 GB, Cache mem 0.001 GB, Peak mem 36.963 GB
Iter 2: Train loss 0.574, Learning Rate 1.000e-05, It/sec 0.008, Tokens/sec 3.132, Trained Tokens 607, Active mem 1.509 GB, Cache mem 0.002 GB, Peak mem 36.963 GB
Iter 3: Train loss 0.511, Learning Rate 1.000e-05, It/sec 0.008, Tokens/sec 1.767, Trained Tokens 820, Active mem 1.509 GB, Cache mem 0.002 GB, Peak mem 36.963 GB
Iter 4: Train loss 0.495, Learning Rate 1.000e-05, It/sec 0.005, Tokens/sec 2.656, Trained Tokens 1308, Active mem 1.507 GB, Cache mem 0.001 GB, Peak mem 44.920 GB
Iter 5: Train loss 0.538, Learning Rate 1.000e-05, It/sec 0.008, Tokens/sec 1.478, Trained Tokens 1484, Active mem 1.509 GB, Cache mem 0.002 GB, Peak mem 44.920 GB
I

In [ ]:
# Plot the training curve from the resumable trainer history.
import matplotlib.pyplot as plt
import pandas as pd

history_rows = []
if TRAINING_HISTORY_PATH.exists():
    with TRAINING_HISTORY_PATH.open("r", encoding="utf-8") as handle:
        history_rows = [json.loads(line) for line in handle if line.strip()]

train_df = pd.DataFrame([row for row in history_rows if row.get("kind") == "train"])
val_df = pd.DataFrame([row for row in history_rows if row.get("kind") == "validation"])

if train_df.empty:
    print("No training loss rows found yet at:", TRAINING_HISTORY_PATH)
else:
    display(train_df.tail(10))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(train_df["iteration"], train_df["train_loss"], marker="o", label="train loss")
    if not val_df.empty:
        ax.plot(val_df["iteration"], val_df["val_loss"], marker="s", label="validation loss")
    ax.set_title("MLX LoRA Training Loss")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

    memory_columns = [column for column in ["active_memory_gb", "cache_memory_gb", "peak_memory_gb"] if column in train_df]
    if memory_columns:
        fig, ax = plt.subplots(figsize=(9, 4))
        for column in memory_columns:
            ax.plot(train_df["iteration"], train_df[column], marker=".", label=column)
        ax.set_title("MLX Memory During Training")
        ax.set_xlabel("Iteration")
        ax.set_ylabel("GB")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()


## Save Training Metadata


In [ ]:
TRAINING_METADATA_PATH = SFT_WORK_DIR / "training_metadata.json"
training_metadata = {
    "student_base_model": STUDENT_MODEL,
    "teacher_model": TEACHER_MODEL,
    "sft_source_path": str(SFT_SOURCE_PATH),
    "mlx_data_dir": str(MLX_DATA_DIR),
    "adapter_output_dir": str(ADAPTER_OUTPUT_DIR),
    "adapter_config_path": str(ADAPTER_CONFIG_PATH),
    "checkpoint_root": str(CHECKPOINT_ROOT),
    "training_history_path": str(TRAINING_HISTORY_PATH),
    "resume_mode": RESUME_TRAINING,
    "raw_rows": len(rows),
    "rows_kept": len(rows_that_fit),
    "rows_dropped_over_length": len(rows_too_long),
    "length_stats": length_stats,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_num_layers": LORA_NUM_LAYERS,
    "lora_rank": LORA_RANK,
    "lora_scale": LORA_SCALE,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUMULATION_STEPS,
    "iters": ITERS,
    "checkpoint_every": CHECKPOINT_EVERY,
    "steps_per_eval": STEPS_PER_EVAL,
    "steps_per_report": STEPS_PER_REPORT,
    "training_seed": TRAINING_SEED,
    "grad_checkpoint": GRAD_CHECKPOINT,
    "validation_task_fraction": VALIDATION_TASK_FRACTION,
    "split_seed": SPLIT_SEED,
    "run_training": RUN_TRAINING,
    "training_result": training_result,
    "adapter_exists": (ADAPTER_OUTPUT_DIR / "adapters.safetensors").exists(),
    "adapter_config_exists": ADAPTER_CONFIG_PATH.exists(),
}
SFT_WORK_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_METADATA_PATH.write_text(json.dumps(cfg.make_json_safe(training_metadata), indent=2), encoding="utf-8")
print(json.dumps(cfg.make_json_safe(training_metadata), indent=2)[:4000])
print("Saved metadata to:", TRAINING_METADATA_PATH)


## Smoke Test Adapter Loading

This does not replace the held-out benchmark eval. It only checks that MLX-LM can load the trained adapter and generate from one training-shaped prompt.


In [ ]:
from mlx_lm import generate as mlx_generate
from mlx_lm import load as mlx_load
from mlx_lm.sample_utils import make_sampler

if not (ADAPTER_OUTPUT_DIR / "adapters.safetensors").exists():
    raise FileNotFoundError(ADAPTER_OUTPUT_DIR / "adapters.safetensors")

smoke_row = validation_rows[0]
model, mlx_tokenizer = mlx_load(STUDENT_MODEL, adapter_path=str(ADAPTER_OUTPUT_DIR))
smoke_prompt = mlx_tokenizer.apply_chat_template(
    smoke_row["messages"][:-1],
    tools=smoke_row.get("tools"),
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
smoke_text = mlx_generate(
    model,
    mlx_tokenizer,
    prompt=smoke_prompt,
    max_tokens=128,
    sampler=make_sampler(temp=0.0, top_p=1.0, top_k=0),
    verbose=False,
)
print("Expected teacher target:\n")
print(smoke_row["target_text"][:1000])
print("\nStudent with MLX adapter generated:\n")
print(smoke_text)


## What This Notebook Produces

- MLX-LM train data: printed as `MLX_DATA_DIR` above.
- MLX LoRA adapter: printed as `ADAPTER_OUTPUT_DIR` above.
- Full training checkpoints: printed as `CHECKPOINT_ROOT` above.
- Loss and memory history: printed as `TRAINING_HISTORY_PATH` above.
- Metadata: printed as `TRAINING_METADATA_PATH` above.

Notebook 05 loads the adapter and runs the held-out τ³-Bench retail eval again. If training is interrupted, set `RESUME_TRAINING = "latest"` and rerun the training cell to continue from the last full checkpoint with optimizer state and iteration count restored. Keep the batch size and training config the same when resuming.
